## LSTM Model Explanations

### Introduction

The most complex model we built during the development of Urban Gala was the LSTM inspired by our Latent Population Simulation research. As we are restricted to 1GB RAM for our EC2 instance and 2GB per push for GitHub, most of the complexity is preprocessed and handled off line. However, in the paper, although the results are explained, there is not a high enough word count to fully explain the model. This paper shows the base class for the latent population simulation and explains all parameters, callbacks and sechedulers. In the explanation we assume familiarity with time series modelling and its mathematical motivation.

### LSTM

In [ ]:


class LatentPopulationSimulation:
    def __init__(self, sequence_length=14, lstm_units=128, forecast_horizon=12):
        """
        Latent Population Simulation Model
        """
        self.sequence_length = sequence_length
        self.lstm_units = lstm_units
        self.forecast_horizon = forecast_horizon
        self.model = None
        self.feature_scaler = RobustScaler()
        self.zone_ids = None
        self.n_zones = None
        self.feature_names = None
        
    def prepare_data(self, df, validation_split=0.2):
        """
        Prepare merged_df for LSTM training
        """

        df = df.copy()
        
      
        if 'day' in df.columns:
            df = df.sort_values('day').reset_index(drop=True)
        
        net_columns = [col for col in df.columns if 'NET' in col]
        self.zone_ids = []
        for col in net_columns:
            zone_id = col.replace(' NET', '')
            self.zone_ids.append(zone_id)
        
        self.n_zones = len(net_columns)        
        population_distributions = self._calculate_population_distribution(df)        
        features = self._engineer_features(df)        

        X, y = self._create_sequences(features, population_distributions)
        
        split_idx = int(len(X) * (1 - validation_split))
        X_train, X_val = X[:split_idx], X[split_idx:]
        y_train, y_val = y[:split_idx], y[split_idx:]
        
        
        return X_train, X_val, y_train, y_val
    
    def _calculate_population_distribution(self, df):
        """
        Calculate population distribution from mobility data
        
        """
        pu_columns = [col for col in df.columns if 'PU' in col and 'NET' not in col]
        dof_columns = [col for col in df.columns if 'DOF' in col and 'NET' not in col]
        net_columns = [col for col in df.columns if 'NET' in col]
        
        # Activity per zone
        activity_by_zone = np.zeros((len(df), self.n_zones))
        
        for i, zone_id in enumerate(self.zone_ids):
            pu_col = f"{zone_id} PU"
            dof_col = f"{zone_id} DOF"
            
            if pu_col in df.columns and dof_col in df.columns:
                activity_by_zone[:, i] = df[pu_col] + df[dof_col]
        
        # Proportions sum to one
        population_distributions = np.zeros_like(activity_by_zone)
        
        for i in range(len(activity_by_zone)):
            total_activity = np.sum(activity_by_zone[i])
            if total_activity > 0:
                population_distributions[i] = activity_by_zone[i] / total_activity
            else:
                # If no activity, assume uniform distribution (This is not advised directly by research papers, I just think this is a logical thing to do)
                population_distributions[i] = np.ones(self.n_zones) / self.n_zones
        
        # Smoothing
        population_distributions = self._smooth_distributions(population_distributions)
        
        return population_distributions
    
    def _smooth_distributions(self, distributions, window=3):
        """Apply rolling average to smooth population distributions"""
        smoothed = np.copy(distributions)
        
        for i in range(self.n_zones):
            zone_series = pd.Series(distributions[:, i])
            smoothed[:, i] = zone_series.rolling(window=window, center=True, min_periods=1).mean()
        
        # Need to renormalise
        for i in range(len(smoothed)):
            total = np.sum(smoothed[i])
            if total > 0:
                smoothed[i] = smoothed[i] / total
        
        return smoothed
    
    def _engineer_features(self, df):
        """
        Create features from your merged_df columns
        Feature semantics follow their naming
        """
        
        pu_columns = [col for col in df.columns if 'PU' in col and 'NET' not in col]
        dof_columns = [col for col in df.columns if 'DOF' in col and 'NET' not in col]
        net_columns = [col for col in df.columns if 'NET' in col]
        
        print(f"Found {len(pu_columns)} PU columns, {len(dof_columns)} DOF columns, {len(net_columns)} NET columns")
        
        df['total_pickups'] = df[pu_columns].sum(axis=1)
        df['total_dropoffs'] = df[dof_columns].sum(axis=1)
        df['total_activity'] = df['total_pickups'] + df['total_dropoffs']
        df['total_net_flow'] = df[net_columns].sum(axis=1)
        df['flow_balance_ratio'] = np.where(df['total_pickups'] > 0, 
                                          df['total_dropoffs'] / df['total_pickups'], 1.0)
        
        zone_features = []
        for zone_id in self.zone_ids:
            pu_col = f"{zone_id} PU"
            dof_col = f"{zone_id} DOF"
            
            if pu_col in df.columns and dof_col in df.columns:
                # Zone activity as fraction of total
                activity_ratio_col = f'activity_ratio_{zone_id}'
                zone_activity = df[pu_col] + df[dof_col]
                df[activity_ratio_col] = np.where(df['total_activity'] > 0,
                                                zone_activity / df['total_activity'], 0)
                zone_features.append(activity_ratio_col)
        
        temporal_features = []
        if 'day' in df.columns:
            try:
                df['day_parsed'] = pd.to_datetime(df['day'], errors='coerce')
                df['day_of_week'] = df['day_parsed'].dt.dayofweek
                df['day_of_month'] = df['day_parsed'].dt.day
                df['month'] = df['day_parsed'].dt.month
                df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
                df['is_monday'] = (df['day_of_week'] == 0).astype(int)
                df['is_friday'] = (df['day_of_week'] == 4).astype(int)
                
                # Cyclical encoding for better temporal representation
                df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
                df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
                df['day_of_month_sin'] = np.sin(2 * np.pi * df['day_of_month'] / 31)
                df['day_of_month_cos'] = np.cos(2 * np.pi * df['day_of_month'] / 31)
                
                temporal_features = ['day_of_week', 'is_weekend', 'is_monday', 'is_friday',
                                   'day_of_week_sin', 'day_of_week_cos', 
                                   'day_of_month_sin', 'day_of_month_cos', 'month']
                print("Created temporal features")
            except Exception as e:
                print(f"Could not parse day column: {e}")
        
        lag_features = []
        for lag in [1, 2, 7]:  # 1 day, 2 days, 1 week ago
            lag_col = f'total_activity_lag{lag}'
            df[lag_col] = df['total_activity'].shift(lag)
            lag_features.append(lag_col)
        
        rolling_features = []
        for window in [3, 7]:
            ma_col = f'activity_ma{window}'
            df[ma_col] = df['total_activity'].rolling(window, min_periods=1).mean()
            rolling_features.append(ma_col)
        
        external_features = []
        if 'Temperature' in df.columns:
            external_features.append('Temperature')
        if 'Wind Speed' in df.columns:
            external_features.append('Wind Speed')
        if 'isHoliday' in df.columns:
            external_features.append('isHoliday')
        if 'isWeekday' in df.columns:
            external_features.append('isWeekday')
        
        # Combine features
        base_features = ['total_pickups', 'total_dropoffs', 'total_activity', 
                        'total_net_flow', 'flow_balance_ratio']
        
        all_features = (base_features + zone_features + temporal_features + 
                       lag_features + rolling_features + external_features)
        
        # Drop any features that don't exist
        all_features = [col for col in all_features if col in df.columns]
        self.feature_names = all_features
        
        print(f"Total features selected: {len(all_features)}")
        
        feature_data = df[all_features].fillna(method='ffill').fillna(0)
        
        feature_scaled = self.feature_scaler.fit_transform(feature_data)
        
        return feature_scaled
    
    def _create_sequences(self, features, targets):
        """
        Create sequences for LSTM
        This is the input LPS from DNNs, as referenced in the report
        """
        X, y = [], []
        
        for i in range(self.sequence_length, len(features) - self.forecast_horizon + 1):
            X.append(features[i - self.sequence_length:i])
            y.append(targets[i:i + self.forecast_horizon])
            
        return np.array(X), np.array(y)
    
    def build_model(self, n_features):
        """
        Build the LSTM model for sequence to sequence prediction
        A simplification of this architecture is described in the paper
        """
        self.model = Sequential([
            Bidirectional(LSTM(self.lstm_units, return_sequences=True), 
                         input_shape=(self.sequence_length, n_features)),
            BatchNormalization(),
            Dropout(0.3),
            
            Bidirectional(LSTM(self.lstm_units // 2, return_sequences=False)),
            BatchNormalization(),
            Dropout(0.3),
            
            Dense(self.lstm_units, activation='relu'),
            Dropout(0.2),
            
            tf.keras.layers.RepeatVector(self.forecast_horizon),
            
            LSTM(self.lstm_units // 2, return_sequences=True),
            BatchNormalization(),
            Dropout(0.2),
            
            LSTM(self.lstm_units // 4, return_sequences=True),
            BatchNormalization(),
            Dropout(0.2),
            
            TimeDistributed(Dense(self.n_zones, activation='softmax'))
        ])
        
        self.model.compile(
            optimizer=Adam(learning_rate=0.001, clipnorm=1.0),
            loss='categorical_crossentropy',
            metrics=['mae']
        )
        
        return self.model
    
    def train(self, X_train, y_train, X_val, y_val, epochs=100):
        """Train the model"""
        if self.model is None:
            self.build_model(X_train.shape[2])
        
        # for information on callback see report
        callbacks = [
            EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-7, verbose=1)
        ]
        
        history = self.model.fit(
            X_train, y_train,
            epochs=epochs,
            batch_size=32,
            validation_data=(X_val, y_val),
            callbacks=callbacks,
            verbose=1
        )
        
        return history
    
    def predict(self, X):
        """Make predictions"""
        return self.model.predict(X, batch_size=32)
    
    def forecast_next_period(self, recent_data, n_days=None):
        """
        Forecast population distribution for the next n_days
        """
        if n_days is None:
            n_days = self.forecast_horizon
            
        # Prepare the input data
        features = self._engineer_features(recent_data)
        
        # Take the last sequence_length days
        if len(features) >= self.sequence_length:
            X_input = features[-self.sequence_length:].reshape(1, self.sequence_length, -1)
            predictions = self.predict(X_input)
            return predictions[0][:n_days]  # Return only n_days predictions
        else:
            raise ValueError(f"Need at least {self.sequence_length} days of recent data")
    
    def evaluate_predictions(self, y_true, y_pred):
        """Evaluate model performance"""
        y_true_flat = y_true.reshape(-1, self.n_zones)
        y_pred_flat = y_pred.reshape(-1, self.n_zones)
        
        mae_per_zone = np.mean(np.abs(y_true_flat - y_pred_flat), axis=0)
        overall_mae = np.mean(mae_per_zone)
        
        # Check if predictions sum to 1 (they should for probability distributions, v important)
        pred_sums = np.sum(y_pred_flat, axis=1)
        avg_sum = np.mean(pred_sums)
        sum_std = np.std(pred_sums)
        
        # Zone correlations
        correlations = []
        for zone in range(self.n_zones):
            if np.std(y_true_flat[:, zone]) > 0:
                corr, _ = pearsonr(y_true_flat[:, zone], y_pred_flat[:, zone])
                if not np.isnan(corr):
                    correlations.append(corr)
        
        avg_correlation = np.mean(correlations) if correlations else 0
        
        return {
            'mae_per_zone': mae_per_zone,
            'overall_mae': overall_mae,
            'prediction_sum_mean': avg_sum,
            'prediction_sum_std': sum_std,
            'average_correlation': avg_correlation,
            'zone_correlations': correlations,
            'n_zones': self.n_zones
        }
    
    def plot_sample_predictions(self, y_true, y_pred, sample_idx=0, n_zones_to_show=10):
        """Plot sample predictions vs actual"""
        fig, axes = plt.subplots(2, 1, figsize=(15, 10))

        zones_to_show = min(n_zones_to_show, self.n_zones)
        

        for i in range(zones_to_show):
            axes[0].plot(y_true[sample_idx, :, i], label=f'Actual Zone {self.zone_ids[i]}', alpha=0.7)
            axes[1].plot(y_pred[sample_idx, :, i], label=f'Predicted Zone {self.zone_ids[i]}', alpha=0.7)
        
        axes[0].set_title('Actual Population Distribution Over Time')
        axes[0].set_ylabel('Proportion')
        axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        axes[0].grid(True, alpha=0.3)
        
        axes[1].set_title('Predicted Population Distribution Over Time')
        axes[1].set_xlabel('Time Step')
        axes[1].set_ylabel('Proportion')
        axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()


# ==== MAIN IMPLEMENTATION FOR YOUR merged_df ====

def run_population_forecasting(merged_df, forecast_horizon=12):
    """
    Pipeline
    Initialise -> Prepare data -> Train model -> Make Predictions -> Evaluate
    """
    
    forecaster = PopulationDistributionForecaster(
        sequence_length=14, 
        lstm_units=128,
        forecast_horizon=forecast_horizon
    )
    
    X_train, X_val, y_train, y_val = forecaster.prepare_data(merged_df, validation_split=0.2)
    
    history = forecaster.train(X_train, y_train, X_val, y_val, epochs=100)
    
    predictions = forecaster.predict(X_val)
    
    metrics = forecaster.evaluate_predictions(y_val, predictions)
    
    # See report for a summarised, focused interpretation of these results
    print(f"*****RESULTS*****")
    print(f"Overall MAE: {metrics['overall_mae']:.4f}")
    print(f"Average Correlation: {metrics['average_correlation']:.4f}")
    print(f"Predictions sum (mean): {metrics['prediction_sum_mean']:.4f} (should be ~1.0)")
    print(f"Predictions sum (std): {metrics['prediction_sum_std']:.4f} (should be ~0.0)")
    

    
    return forecaster, history, predictions, metrics


#### **Model Architecture**

The forecaster uses a encoder-decoder LSTM architecture with the following specs:<br>
Input Layer: (sequence_length=14, n_features=variable)<br>
├── Bidirectional LSTM(128 units, return_sequences=True)<br>
├── BatchNormalization()<br>
├── Dropout(0.3)<br>
├── Bidirectional LSTM(64 units, return_sequences=False)<br>
├── BatchNormalization()<br>
├── Dropout(0.3)<br>
├── Dense(128 units, ReLU activation)<br>
├── Dropout(0.2)<br>
├── RepeatVector(forecast_horizon=12)<br>
├── LSTM(64 units, return_sequences=True)<br>
├── BatchNormalization()<br>
├── Dropout(0.2)<br>
├── LSTM(32 units, return_sequences=True)<br>
├── BatchNormalization()<br>
├── Dropout(0.2)<br>
└── TimeDistributed(Dense(n_zones, softmax activation))<br>

The encoder employs bidirectional LSTM layers to capture temporal dependencies in both forward and backward directions. This architecture enables the model to leverage future context when processing historical sequences, addressing the limitation of unidirectional RNNs that only consider past information at each timestep.

The mathematical formulation for bidirectional processing:

Forward LSTM: h_t^f = LSTM_f(x_t, h_{t-1}^f)<br>
Backward LSTM: h_t^b = LSTM_b(x_t, h_{t+1}^b)<br>
Combined representation: h_t = [h_t^f; h_t^b]<br>

The decoder architecture utilizes RepeatVector to broadcast the encoded representation across the forecast horizon, followed by stacked LSTM layers that generate sequential predictions. This approach maintains "temporal coherence" (meaning consistent and meaningful information over time) across the 12-day forecast window while preserving the probabilistic nature of population distributions.

#### **Data Engineering Pipeline**
In this section we assume good familiarity with time series modelling.

Population Distribution Calculation:

Zone-level population distributions are derived from mobility indicators using activity proxies:
Activity_i(t) = PU_i(t) + DOF_i(t)<br>
P_i(t) = Activity_i(t) / Σ(Activity_j(t)) for j ∈ Zones<br>
Where:

PU_i(t): Pickups in zone i at time t<br>
DOF_i(t): Drop-offs in zone i at time t<br>
P_i(t): Normalized population proportion in zone i at time t<br>

Smoothing:
A rolling window smoothing operation (window=3) is applied to mitigate high-frequency noise while preserving underlying temporal patterns:
P_smooth_i(t) = (1/w) * Σ(P_i(t+k)) for k ∈ [-floor(w/2), floor(w/2)]<br>
Post-smoothing renormalization ensures Σ(P_smooth_i(t)) = 1.0 ∀t.

Feature Engineering:
The feature engineering pipeline constructs a comprehensive feature space encompassing:

Aggregate Mobility Metrics:

Total system throughput: Σ(PU_i + DOF_i)<br>
Net flow dynamics: Σ(NET_i)<br>
Flow balance ratio: Σ(DOF_i) / Σ(PU_i)<br>

Zone-Specific Features:

Activity ratio per zone: (PU_i + DOF_i) / Σ(PU_j + DOF_j)<br>
Relative zone importance metrics<br>

Temporal Encodings:

Cyclical time representations using sine/cosine transformations<br>
Day-of-week encoding: sin(2π * dow / 7), cos(2π * dow / 7)<br>
Day-of-month encoding: sin(2π * dom / 31), cos(2π * dom / 31)<br>
Binary temporal indicators (weekend, Monday, Friday flags)<br>

Lagged Features:

Activity lags at t-1, t-2, and t-7 (capturing daily and weekly dependencies)<br>
Rolling averages over 3-day and 7-day windows<br>


Data Scaling and Normalization:

RobustScaler is employed for feature normalization, providing superior performance compared to StandardScaler when dealing with mobility data containing outliers:

X_scaled = (X - median(X)) / IQR(X)<br>

This approach ensures robustness against extreme events (protests, emergencies, major events) that could skew standard normalization methods.

#### **Training**
Loss Function and Optimization:<br>
The model employs categorical cross-entropy loss, appropriate for probability distribution targets:<br>
L = -Σ(y_true * log(y_pred))<br>
Adam optimizer with gradient clipping (clipnorm=1.0) prevents exploding gradients common in RNN training. Initial learning rate: 0.001. This is explained in context in the report

Regularization Strategy:<br>

Dropout Regularization: Implemented at multiple layers with rates 0.2-0.3 to prevent overfitting while maintaining model capacity.
Batch Normalization: Applied after LSTM layers to stabilize training dynamics and accelerate convergence.<br>

Temporal Validation Split: Chronological splitting ensures temporal integrity, with 80% training data and 20% validation data maintaining temporal order.

#### **Training Callbacks and Scheduling**

The EarlyStopping callback implements validation-based training termination:<br>
Parameters:<br>

Monitor: validation loss<br>
Patience: 15 epochs<br>
Restore best weights: True<br>

Rationale: Prevents overfitting by terminating training when validation performance plateaus, automatically reverting to the best-performing model state. This is very important for time series models where overfitting can severely degrade generalization performance.<br>

Adaptive Learning Rate Scheduling:<br>
ReduceLROnPlateau implements dynamic learning rate adjustment:<br>
Parameters:<br>

Monitor: validation loss<br>
Reduction factor: 0.5<br>
Patience: 8 epochs<br>
Minimum learning rate: 1e-7<br>

Mechanism: When validation loss fails to improve for 8 consecutive epochs, the learning rate is halved. This adaptive scheduling enables:<br>

Rapid initial convergence with higher learning rates
Fine-grained optimization in later stages with reduced learning rates
Escape from local minima through learning rate reductions<br>

Mathematical Justification: The learning rate schedule follows:
lr(t+1) = max(lr(t) * 0.5, 1e-7) if no improvement for 8 epochs<br>

#### **Sequence Generation and Temporal Dependencies**
Sliding Window Approach: (most common approach I found)
The model generates training sequences using a sliding window methodology:<br>
For dataset length T, sequence length L=14, and forecast horizon H=12:<br>

Input sequences: X[i] = features[i-L:i] for i ∈ [L, T-H+1]<br>
Target sequences: y[i] = distributions[i:i+H] for i ∈ [L, T-H+1]<br>

This generates overlapping sequences that capture temporal continuity while providing sufficient training examples.

Temporal Dependency Modeling:<br>
The 14-day input window captures:<br>

Short-term dependencies: Daily patterns and day-to-day variations
Weekly cycles: Weekday vs. weekend patterns<br>
Bi-weekly patterns: Potential fortnightly cycles in urban activity<br>

The 12-day forecast horizon provides actionable medium-term predictions while maintaining acceptable accuracy degradation.

#### **Output Processing and Constraints**
Probability Distribution Enforcement:
The softmax activation function ensures valid probability distributions:<br>
P_i(t) = exp(z_i(t)) / Σ(exp(z_j(t))) for j ∈ Zones
This guarantees:<br>

P_i(t) ∈ [0, 1] ∀i, t<br>
Σ(P_i(t)) = 1.0 ∀t<br>

Multi-Zone Consistency:<br>
The TimeDistributed wrapper applies the same dense transformation across all forecast timesteps, ensuring temporal consistency in the output generation process.

#### **Performance Evaluation**
Quantitative Metrics:<br>
Mean Absolute Error (MAE):<br>
MAE = (1/N*T*Z) * Σ|y_true - y_pred|<br>
Zone-Level Correlation Analysis:<br>
Pearson correlation coefficients computed per zone to assess pattern fidelity:<br>
ρ_i = corr(y_true_i, y_pred_i) for zone i<br>
Distribution Validity Checks:<br>

Prediction sum mean (should approximate 1.0)<br>
Prediction sum standard deviation (should approximate 0.0)<br>

Temporal Consistency Validation:<br>
The evaluation framework verifies that predictions maintain temporal smoothness and do not show unrealistic discontinuities between forecast timesteps.<br>


#### **Computational Complexity and Scalability**
For completeness I will include the time and space complexity of the algorithm.<br>
Time complexity: O(T * L * U²) where T is sequence count, L is sequence length, and U is LSTM unit count.<br>
Space complexity: O(B * L * F) for batch size B, sequence length L, and feature count F.<br>

Inference Efficiency:<br>
Single forecast generation: O(L * U²) per prediction, enabling real-time deployment within Urban Gala's operational requirements.

#### *Conclusion*
This LSTM-based forecasting system provides Urban Gala with a robust, scalable solution for Manhattan zone busyness prediction. The architecture's bidirectional processing, regularization, and adaptive training callbacks ensure reliable performance across diverse temporal patterns while maintaining the mathematical constraints required for probability distribution forecasting.
The model's sequence-to-sequence approach captures the complex temporal dependencies inherent in urban mobility patterns, while the comprehensive feature engineering pipeline incorporates both endogenous mobility signals and exogenous environmental factors. This combination enables accurate medium-term forecasting essential for Urban Gala's Manhattan operations.